# Att hitta efterfrågedrivkrafter med PROC GLMSELECT: Stegvist urval, LASSO och validerat framåtval

## Sammanfattning

Ett detaljhandelsanalysteam använder **PROC GLMSELECT** för att upptäcka vilka marknadsförings- och prishandtag som faktiskt påverkar veckovis enhetsförsäljning, och skiljer verkliga efterfrågedrivkrafter från brus. Stegvist urval poängsatt med SBC, LASSO med 5-faldig korsvalidering och en framåtsökning validerad på kvarhållna data återfinner alla **samma uppsättning verkliga drivkrafter** — eget pris, konkurrentpris, annonsutgifter, e-postvolym, kampanjer, helgdagar, regionen Nordöst och kanalen Online — och var och en av de fyra planterade lockbetena (`temp_f`, `noise1`, `noise2`, `noise3`) sorteras automatiskt bort.

Metoderna är också väl överens om storleksordningarna: var och en skattar en egenpriseffekt nära **-28 enheter per dollar** och en konkurrentpriseffekt nära **+14**, de substitutionstecken som den datagenererande ekvationen byggde in. Den enda oenigheten finns i marginalen — den korsvaliderade LASSO behåller dessutom en liten, statistiskt försumbar `Region=Midwest`-kontrast (skattning -8,3 med ett standardfel på 8,3) som både stegvist urval och framåtval utesluter. En drivkraftslista som överlever tre olika urvalsfilosofier är betydligt mer pålitlig än en som är finjusterad efter en enda regel.

## Datakällor

All data i denna notebook är **syntetisk** och genereras direkt i koden (inga externa filer, frö `20260531`). Den efterliknar en säsong av butiks-veckopaneler för efterfrågan hos en dagligvaruhandlare.

| Dataset | Rader | Granularitet | Nyckelvariabler |
|---------|------|-------|---------------|
| `demand` | 100 | Butik-vecka | `units` (respons: sålda enheter per vecka); `price`, `competitor` (eget och konkurrerande hyllpris); `adspend`, `email` (mediepåverkan); `promo`, `holiday` (händelseflaggor); `region`, `channel` (CLASS-effekter); `temp_f`, `noise1`-`noise3` (lockbete/irrelevanta prediktorer) |

`units` byggs upp från en känd linjär ekvation så att den "korrekta" uppsättningen drivkrafter kan verifieras; `temp_f` och de tre `noise`-kolumnerna bär ingen verklig signal och finns för att testa om varje urvalsmetod sorterar bort dem.

# Att hitta efterfrågedrivkrafter med PROC GLMSELECT

En kategorichef har dussintals kandidatförklaringar till veckoförsäljningen: eget pris, konkurrentpris, annonsutgifter, e-postvolym, kampanjer, helgdagar, butiksregion, säljkanal, till och med vädret. Att kasta in alla i en enda regression inbjuder till överanpassning och instabila koefficienter. **PROC GLMSELECT** automatiserar sökningen efter en sparsam modell och stöder stegvis, LASSO, elastic net och minsta-vinkel-urval med inbyggd korsvalidering och uppdelning i kvarhållna data.

I den här notebooken gör vi följande:

1. Genererar en realistisk syntetisk butiks-veckopanel för efterfrågan med en *känd* uppsättning verkliga drivkrafter (plus avsiktliga lockbetesvariabler).
2. Kör **stegvist urval** poängsatt med SBC.
3. Kör **LASSO** med 5-faldig korsvalidering.
4. Kör **framåtval validerat på 30 % kvarhållna data**.

En bra urvalsmetod bör återfinna de verkliga drivkrafterna och sortera bort bruset — låt oss se.

## 1. Generera en syntetisk efterfrågepanel

Responsvariabeln `units` konstrueras från en explicit linjär ekvation, så vi känner till facit: pris och konkurrentpris, annonsutgifter, e-postvolym, kampanj- och helgdagsflaggorna samt region- och kanaleffekterna spelar alla roll. Variablerna `temp_f`, `noise1`, `noise2` och `noise3` är rena lockbeten utan något verkligt samband med försäljningen. Ett frö via `call streaminit` gör datan reproducerbar.

In [1]:
/* ---------------------------------------------------------------
   Syntetisk butiks-veckopanel för efterfrågan hos en dagligvaruhandlare.
   'units' följer en KÄND ekvation; temp_f och noise1-3 är lockbeten.
   --------------------------------------------------------------- */
data demand;
    CALL streaminit(20260531);
    LÄNGD region $9 channel $8 promo $3;
    GÖR store_week = 1 TILL 100;
        /* Regionfördelning */
        u1 = rand('uniform');
        OM u1 < 0.34 SÅ region = 'Northeast';
        ANNARS OM u1 < 0.67 SÅ region = 'Midwest';
        ANNARS region = 'South';

        /* Säljkanal */
        OM rand('uniform') < 0.45 SÅ channel = 'Store';
        ANNARS channel = 'Online';

        /* Pris- och mediedrivkrafter */
        price      = round(8  + rand('uniform') * 12, 0.01);
        competitor = round(8  + rand('uniform') * 12, 0.01);
        adspend    = round(rand('gamma', 2) * 500, 1);
        email      = round(rand('uniform') * 100, 1);

        /* Händelseflaggor och ett irrelevant väderlockbete */
        temp_f     = round(40 + rand('uniform') * 50, 0.1);
        holiday    = (rand('uniform') < 0.12);
        OM rand('uniform') < 0.40 SÅ promo = 'Yes';
        ANNARS promo = 'No';

        /* Rena brusprediktorer (ingen verklig effekt) */
        noise1 = rand('normal');
        noise2 = rand('normal');
        noise3 = rand('normal');

        /* Sålda enheter per vecka från en känd strukturell ekvation */
        units = 900
              - 28   * price
              + 14   * competitor
              + 0.06 * adspend
              + 1.2  * email
              + 55   * (promo = 'Yes')
              + 70   * holiday
              + 40   * (region = 'Northeast')
              + 25   * (channel = 'Online')
              + 30   * rand('normal');
        OM units < 0 SÅ units = 0;
        UTDATA;
    SLUT;
    BEHÅLL region channel promo price competitor adspend email temp_f
         holiday noise1 noise2 noise3 units;
KÖR;



NOTE: DATA demand


NOTE: Wrote demand (100 rows, 13 columns).
NOTE: DATA elapsed:
  wall  0.04 seconds
  cpu   0.04 seconds


## 2. Profilera datan

Innan modellering, bekräfta att responsen och de viktigaste kontinuerliga kandidaterna ligger på rimliga skalor.

In [2]:
PROCEDUR MEDELVÄRDEN data=demand n mean std MIN MAX maxdec=1;
    VARIABEL units price competitor adspend email;
    ETIKETT units="Sålda enheter" price="Pris" competitor="Konkurrentpris"
          adspend="Annonsutgifter" email="E-postvolym";
    TITEL 'Veckovis efterfrågan och kandidatdrivkrafter';
KÖR;


                                      Veckovis efterfrågan och kandidatdrivkrafter                                      

                                                  The MEANS Procedure

 Variable    Label                  N        Mean     Std Dev     Minimum     Maximum
 ------------------------------------------------------------------------------------
 units       Sålda enheter        100       874.8       136.3       508.6      1162.3
 price       Pris                 100        14.0         3.4         8.0        19.9
 competitor  Konkurrentpris       100        13.8         3.4         8.1        19.9
 adspend     Annonsutgifter       100       990.6       726.9        50.0      3358.0
 email       E-postvolym          100        45.5        26.4         0.0        99.0
 ------------------------------------------------------------------------------------




NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## 3. Stegvist urval poängsatt med SBC

Stegvist urval lägger omväxlande till och tar bort effekter, här bedömt av **Schwarz Bayesian Criterion (SBC)** för både inträdestestet (`select=sbc`) och det slutliga modellvalet (`choose=sbc`). SBC straffar komplexitet hårdare än AIC, vilket gynnar magrare modeller.

Viktiga satser och alternativ:

- `CLASS region channel promo / param=reference` deklarerar de kategoriska prediktorerna med referenscellskodning.
- `selection=stepwise(select=sbc choose=sbc)` driver sökningen.
- `details=summary` skriver ut den stegvisa urvalssammanfattningen; `stb` lägger till standardiserade koefficienter så att effekter på olika skalor kan jämföras.
- `output out=demand_scored p=predicted r=residual` sparar anpassade värden och residualer för vidare bedömning.

Läs **Stepwise Selection Summary** som sökningens spår: den startar från den fullständiga 12-effektsmodellen och *tar bort* effekter en i taget, och släpper i tur och ordning `noise1`, `noise2`, `temp_f`, kontrasten `Region=Midwest` och `noise3`, eftersom varje borttagning sänker SBC. Det som återstår i tabellen **Parameter Estimates** är den valda modellen.

In [3]:
PROCEDUR glmselect data=demand seed=20260531;
    KLASS region channel promo / PARAM=reference;
    MODEL units = region channel promo price competitor adspend
                  email temp_f holiday noise1 noise2 noise3
        / selection=stepwise(select=sbc choose=sbc)
          details=summary stb;
    UTDATA out=demand_scored p=predicted r=residual;
    ETIKETT units="Sålda enheter" region="Region" channel="Kanal" promo="Kampanj"
          price="Pris" competitor="Konkurrentpris" adspend="Annonsutgifter"
          email="E-postvolym" temp_f="Temperatur (F)" holiday="Helgdag"
          noise1="Brus1" noise2="Brus2" noise3="Brus3";
    TITEL 'Stegvist urval av efterfrågedrivkrafter (SBC)';
KÖR;


                                      Veckovis efterfrågan och kandidatdrivkrafter                                      

The GLMSELECT Procedure


Dependent Variable: UNITS Sålda enheter


Number of Observations Used: 100

          Class Level Information          

   Class    Levels                   Values
--------  --------  -----------------------
  region         3  Midwest Northeast South
 channel         2             Online Store
   promo         2                   No Yes

                                                Stepwise Selection Summary                                                 

    Step    Action          Effect  Number Effects In  R-Square  Adj R-Sq       AIC      AICC       BIC       SBC      C(p)
--------  --------  --------------  -----------------  --------  --------  --------  --------  --------  --------  --------
       1   Removed           Brus1                 12    0.9507    0.9439  707.0094  711.2420  713.1843  740.8766   12.0136
       2   Re


NOTE: PROC GLMSELECT data=demand

NOTE: The data set WORK.DEMAND_SCORED has 100 observations and 15 variables.
NOTE: PROC GLMSELECT statement used.


## 4. LASSO med korsvalidering

LASSO krymper koefficienter mot noll och utför urval och regularisering samtidigt. Istället för att stanna vid ett fast kriterium låter vi **5-faldig korsvalidering** välja den punkt på LASSO-vägen som ger bäst prestanda utanför stickprovet.

- `selection=lasso(choose=cv stop=none)` spårar hela LASSO-vägen och väljer det CV-optimala steget.
- `cvmethod=random(5)` tilldelar observationer till 5 slumpmässiga delar (folds).

**LASSO Selection Summary** visar i vilken ordning effekter träder in när straffet lättar: `price` först, sedan `adspend`, `competitor`, regionen Nordöst, `email`, `promo` och `holiday` — alla sju verkliga signaler träder in innan något lockbete gör det. Korsvalideringen väljer sedan det steg där PRESS utanför stickprovet är lägst, och den resulterande tabellen **Parameter Estimates** behåller exakt de äkta drivkrafterna (plus en liten `Region=Midwest`-term) medan `temp_f`, `noise1`, `noise2` och `noise3` utesluts, vilka bara träder in helt i slutet av vägen.

In [4]:
PROCEDUR glmselect data=demand seed=20260531;
    KLASS region channel promo / PARAM=reference;
    MODEL units = region channel promo price competitor adspend
                  email temp_f holiday noise1 noise2 noise3
        / selection=lasso(choose=cv stop=none)
          cvmethod=RANDOM(5);
    ETIKETT units="Sålda enheter" region="Region" channel="Kanal" promo="Kampanj"
          price="Pris" competitor="Konkurrentpris" adspend="Annonsutgifter"
          email="E-postvolym" temp_f="Temperatur (F)" holiday="Helgdag"
          noise1="Brus1" noise2="Brus2" noise3="Brus3";
    TITEL 'LASSO med 5-faldig korsvalidering';
KÖR;


                                      Veckovis efterfrågan och kandidatdrivkrafter                                      

The GLMSELECT Procedure


Dependent Variable: UNITS Sålda enheter


Number of Observations Used: 100

  Cross Validation Information   

                   Item     Value
-----------------------  --------
Cross Validation Method    Random
        Number of Folds         5

          Class Level Information          

   Class    Levels                   Values
--------  --------  -----------------------
  region         3  Midwest Northeast South
 channel         2             Online Store
   promo         2                   No Yes

                                                         LASSO Selection Summary                                                          

    Step    Action            Effect  Number Effects In  R-Square  Adj R-Sq       AIC      AICC       BIC       SBC      C(p)        PRESS
--------  --------  ----------------  -----------------  --


NOTE: PROC GLMSELECT data=demand

NOTE: PROC GLMSELECT statement used.


## 5. Framåtval validerat på kvarhållna data

En tredje, kompletterande strategi: bygg modellen med **framåtval** (effekter träder bara in, lämnar aldrig), men stanna där prestandan på ett oberoende kvarhållet stickprov är bäst — vilket skyddar direkt mot överanpassning.

- `partition fraction(validate=0.30)` reserverar slumpmässigt 30 % av raderna för validering, vilket lämnar 70 tränings- och 30 valideringsobservationer.
- `selection=forward(select=aic choose=validate)` lägger till effekter efter AIC men väljer den slutliga modellen efter valideringsstickprovets fel.

Tabellen **Partition Fractions** bekräftar 70/30-uppdelningen. Framåtval fortsätter sedan att lägga till effekter tills valideringsfelet slutar förbättras; de åtta effekterna i den slutliga tabellen **Parameter Estimates** är precis de verkliga drivkrafterna, medan de fyra lockbetena aldrig släpps in. När tre metoder byggda på olika principer landar i samma drivkrafter är modellen betydligt mer sannolikt verklig än en artefakt av en enskild urvalsregel.

In [5]:
PROCEDUR glmselect data=demand seed=20260531;
    KLASS region channel promo / PARAM=reference;
    MODEL units = region channel promo price competitor adspend
                  email temp_f holiday noise1 noise2 noise3
        / selection=forward(select=aic choose=validate);
    partition FRACTION(validate=0.30);
    ETIKETT units="Sålda enheter" region="Region" channel="Kanal" promo="Kampanj"
          price="Pris" competitor="Konkurrentpris" adspend="Annonsutgifter"
          email="E-postvolym" temp_f="Temperatur (F)" holiday="Helgdag"
          noise1="Brus1" noise2="Brus2" noise3="Brus3";
    TITEL 'Framåtval validerat på 30 % kvarhållna data';
KÖR;


                                      Veckovis efterfrågan och kandidatdrivkrafter                                      

The GLMSELECT Procedure


Dependent Variable: UNITS Sålda enheter


Number of Observations Used: 70               
Number of Observations Used for Validation: 30

Partition Fractions 

      Role  Fraction
----------  --------
  Training    0.7000
Validation    0.3000
   Testing    0.0000

          Class Level Information          

   Class    Levels                   Values
--------  --------  -----------------------
  region         3  Midwest Northeast South
 channel         2             Online Store
   promo         2                   No Yes

                                                        Forward Selection Summary                                                         

    Step    Action            Effect  Number Effects In  R-Square  Adj R-Sq       AIC      AICC       BIC       SBC      C(p)        PRESS
--------  --------  ----------------  ----


NOTE: PROC GLMSELECT data=demand

NOTE: PROC GLMSELECT statement used.


## 6. Tolka resultaten

Alla tre urvalsstrategier återfinner **samma uppsättning verkliga efterfrågedrivkrafter** och sorterar bort varje lockbete. Läser vi direkt från tabellerna **Parameter Estimates**:

- **Pris** är den dominerande drivkraften. Dess standardiserade koefficient i stegvis-modellen är **-0,70**, klart störst till beloppet, och den råa lutningen ligger mellan **-27,8** (stegvis och LASSO) och **-28,6** (framåtval) enheter per dollar — nästan exakt de -28 som datan genererades med. **Konkurrentpris** flyttar efterfrågan åt andra hållet med ungefär **+14,4** i alla tre anpassningarna, det substitutionsbeteende en kategorichef förväntar sig.
- **Annonsutgifter** (cirka +0,062 enheter per dollar) och **e-postvolym** (cirka +1,2 enheter per utskick) höjer båda försäljningen, vilket kvantifierar mediereaktionen.
- **Kampanjer** och **helgdagar** bär de största händelseeffekterna: kontrasten `Promo=No` ligger på cirka **-51 till -57** relativt en kampanjvecka, och helgdagslyftet är ungefär **+66 till +76** enheter. **Regionen Nordöst** (cirka +49 till +55) och **kanalen Online** (cirka +24 till +32) bär positiva baslinjeeffekter.
- Avgörande är att varje lockbete — `temp_f`, `noise1`, `noise2`, `noise3` — **sorteras bort** av stegvist och framåtval och utesluts från den CV-valda LASSO-modellen. Varje metod återfann den strukturella signalen och ignorerade bruset.

De tre metoderna är inte identiska byte för byte: stegvist urval (SBC) och det kvarhållna-validerade framåtvalet landar i samma åtta effekter, medan den korsvaliderade LASSO dessutom behåller en liten `Region=Midwest`-kontrast (-8,3, standardfel 8,3) — en skillnad vid brusgolvet snarare än en substantiell oenighet. Att en drivkraftslista överlever stegvis SBC, korsvaliderad LASSO och kvarhållna-validerat framåtval är den verkliga slutsatsen: ett fynd som är robust mot tre olika urvalsfilosofier är betydligt mer trovärdigt än ett som är finjusterat efter ett enda kriterium. Med `OUTPUT OUT=demand_scored` som fångar anpassade värden och residualer sträcker sig samma arbetsflöde naturligt till att poängsätta nästa kvartals planerade pris- och kampanjkalender.